# Spot Selection Notebook
## For sorted meiotic cells
### Elizabeth Finn
### July 8, 2025

This notebook will iteratively display segmented and sorted cells, allowing users to identify spots within them by clicking, and output to a file the positions of those spots.

Setup and loading tools:

In [ ]:
ANALYSIS_PREFIX = "20241022_MB"
MICROSCOPE = "SortedLSM"
USER = "butlerm"
DAPI_CHANNEL = 1

In [ ]:
# Set filetype environment variable here, before anything is loaded... "LSM" is LSM, "IMX" is default and for ImageXpress Micro
import os
os.environ["FILE_TYPE"] = MICROSCOPE
os.environ["USER"] = USER

In [ ]:
%matplotlib inline

#import some tools that we will use:
# Utilities
import json
import random
import numpy
import csv
from pathlib import Path
import ipywidgets
from matplotlib import pyplot

#Sci-kit image and matplotlib
import skimage.io
import skimage.measure
import skimage.feature
import skimage.segmentation
from skimage.morphology import disk
import scipy.ndimage
import matplotlib.pyplot as plt
import re

#Our bespoke tools
from models.swarm_job import SwarmJob, RunStrategy
from models.image_filename_glob import ImageFilenameGlob
from models.image_filename import ImageFilename
from models.generate_spot_positions_config import GenerateSpotPositionsConfig
from generate_spot_positions import GenerateSpotPositionsJob
from generate_all_spot_positions import GenerateAllSpotPositionsJob
from generate_all_spot_result_lines_nomask import GenerateAllSpotResultLinesJob

from generate_spot_results_file import GenerateSpotResultsFileJob

Find nuclei and define subroutines for graphing:

In [ ]:
global i 
i = 0

image_folder = "current"
image_path = Path(image_folder)

channel_ID = DAPI_CHANNEL

image_filename_glob = image_path.rglob(str(ImageFilenameGlob(c = channel_ID, 
                                                             suffix="_maximum_projection_nucleus_????", 
                                                             extension="npy")))

cells_list = ([image_file_path for image_file_path in image_filename_glob])
files_count = len(cells_list)
if files_count == 0:
    print("I didn't find any nuclei, check another field.")
else:
    print("I found", files_count,  "nuclei")
    print(str(cells_list[1]))
    
parsed_cell = ImageFilename.parse(str(cells_list[1].relative_to(image_folder)))
image_filename_glob = ImageFilenameGlob.from_image_filename(parsed_cell, excluding_keys=["c"])
images_filenames = [filename for filename in image_path.rglob(str(image_filename_glob))]

global channels
channels = len(images_filenames)
print("Each cell has images for", channels, "channels")
print([channel for channel in range(channels)])

In [ ]:
def display_button_clicked(b):
    LoG_threshold = LoG_slider.value
    local_contrast_threshold = local_contrast_slider.value
    peak_radius = radius_slider.value
    global_contrast_threshold = global_contrast_slider.value
    global i
    find_spots(i, LoG_threshold, local_contrast_threshold, peak_radius, global_contrast_threshold)
    with image_out:
        image_out.clear_output()
        # plot them
        fig1, axs1 = pyplot.subplots(1,channels)

        for j in range(channels):
            ax = axs1[j]
            ax.imshow(segmenters[j].image)
            labels = len(segmenters[j].spots)
            s = 0
            if j+1 != DAPI_CHANNEL:
                for spot in segmenters[j].spots:
                    circle=plt.Circle((spot[4], spot[3]), 7, color='r', alpha=0.3)
                    ax.add_patch(circle)
                    ax.text(spot[4], spot[3], s)
                    s = s + 1
        fig1.set_size_inches(27,9)
        plt.show(fig1)
    with text_out:
        text_out.clear_output()
        print("Displayed!")

In [ ]:
def find_spots(n, LoG_threshold, local_contrast_threshold, peak_radius, global_contrast_threshold):
    cell = cells_list[n]
    # find all channels relevant to cell (we only have dapi so far)
    parsed_cell = ImageFilename.parse(str(cell.relative_to(image_folder)))
    image_filename_globs = [ImageFilenameGlob(date = parsed_cell.date,
                                              position = parsed_cell.position,
                                              group = parsed_cell.group,
                                              f = parsed_cell.f,
                                              c = channel+1,
                                              suffix = parsed_cell.suffix,
                                              extension = parsed_cell.extension) for channel in range(channels)]
    images_filenames = [next(image_path.rglob(str(image_filename_glob))) for image_filename_glob in image_filename_globs]
    global segmenters
    segmenters = [GenerateSpotPositionsJob(image, "no_output", image_folder,
                                           LoG_threshold, local_contrast_threshold, peak_radius, global_contrast_threshold) 
                  for image in images_filenames]

In [ ]:
def output_selected_spot(channelID, spotID, cellID):
    #make destination filename from nucleus filename and spot ID
    cell = cells_list[cellID]
    parsed_cell = ImageFilename.parse(str(cell.relative_to(image_folder)))
    suffix_base = re.sub("_maximum_projection_", "_", parsed_cell.suffix)
    spot_suffix = "%s_spot_%i"%(suffix_base,spotID)
    spot_filename = "%s/CS%s_g%s/%s_ch%s%s.%s" % (
        parsed_cell.date,
        parsed_cell.position,
        parsed_cell.group,
        parsed_cell.f,
        channelID,
        spot_suffix,
        "npy")
    
    #identify cell parameters (date, group, position, cell_type)
    line = {
        "filename": spot_filename,
        "date": parsed_cell.date,
        "group" : parsed_cell.group,
        "position": parsed_cell.position,
        "field": parsed_cell.f,
        "channel": channelID,
        "nucleus_index": cellID,
        "spot_index": spotID,
    }

    #define spot center
    spot = segmenters[channelID-1].spots[spotID]
    pixel_center = (round(spot[3]), round(spot[4]))

    #look up z center
    z_center_suffix = re.sub("_maximum_projection_", "_z_center_", parsed_cell.suffix)
    z_center_filename = "%s/CS%s_g%s/%s_ch%s%s.%s" % (
        parsed_cell.date,
        parsed_cell.position,
        parsed_cell.group,
        parsed_cell.f,
        channelID,
        z_center_suffix,
        "npy")
    z_center_image = numpy.load("%s/%s"%(image_folder, z_center_filename))
    center_z = z_center_image[pixel_center]
    
    #look up r position
    dt_suffix = re.sub("_maximum_projection_nucleus_", "_dt_", parsed_cell.suffix)
    dt_filename = "%s/CS%s_g%s/%s_ch%s%s.%s" % (
        parsed_cell.date,
        parsed_cell.position,
        parsed_cell.group,
        parsed_cell.f,
        "XX",
        dt_suffix,
        "npy")
    dt_source_folder = re.sub("MIPS", "dts", image_folder)
    dt_image = numpy.load("%s/%s"%(dt_source_folder, dt_filename))
    center_r = dt_image[pixel_center]
    
    #add spot details to dictionary
    line.update({
        "center_x": spot[4],
        "center_y": spot[3],
        "center_z": center_z,
        "center_r": center_r,
        "area": spot[0],
        "eccentricity": spot[1],
        "solidity": spot[2],
        "integrated_intensity": spot[5]
    })

    #output line to csv file
    output_file = "%s_spot_positions.csv"%(line["date"])
    if not(os.path.exists(output_file)) or (os.path.getsize(output_file) == 0):
        with open(output_file, 'w') as csv_file:
            csv_writer = csv.DictWriter(csv_file, line.keys())
            csv_writer.writeheader()
    with open(output_file, 'a') as csv_file:
        csv_writer = csv.DictWriter(csv_file, line.keys())
        csv_writer.writerow(line)
    
    #output spot line formatted as:
    #date	group	position	field	channel	nucleus_index	spot_index	center_x	center_y	center_z	center_r	area	eccentricity	solidity
    print("Result: %s"%(line))

## Interactively find and output spots!

In [ ]:
find_spots(i, 0.1, 1, 2, 0.1)
LoG_slider = ipywidgets.FloatSlider(value = 0.1, min = 0, max = 1, step = 0.05, 
                                    description = "LoG Threshold", continuous_update = True, orientation = 'horizontal')
local_contrast_slider = ipywidgets.FloatSlider(value = 1, min = 1, max = 2, step = 0.05, 
                                               description = "Local Contrast Threshold", continuous_update = True, orientation = 'horizontal')
radius_slider = ipywidgets.IntSlider(value = 2, min = 2, max = 5, step = 1, 
                                     description = "Spot Radius", continuous_update = True, orientation = 'horizontal')
global_contrast_slider=ipywidgets.FloatSlider(value = 0.1, min = 0, max = 1, step = 0.05, 
                                               description = "Global Contrast Threshold", continuous_update = True, orientation = 'horizontal')
sliders = ipywidgets.VBox([radius_slider,
                          LoG_slider,
                          local_contrast_slider,
                          global_contrast_slider])

display_button = ipywidgets.Button(description="Display")
output_button = ipywidgets.Button(description="Save spots")
next_cell_button = ipywidgets.Button(description="Next cell")

channels_text = ipywidgets.Text(placeholder = "Channel (1-indexed)",
                                description = "Channel:")
spot1_text = ipywidgets.Text(placeholder="Spot (as on image)",
                                description = "Spot 1:")
spot2_checked = ipywidgets.Checkbox(value=False,
                                    description="Second spot?")
spot2_text = ipywidgets.Text(placeholder="Spot (as on image)",
                                description = "Spot 2:")


selections = ipywidgets.VBox([channels_text, spot1_text, spot2_checked, spot2_text])
buttons = ipywidgets.VBox([display_button, output_button, next_cell_button])
text_out = ipywidgets.Output()
image_out = ipywidgets.Output()

controls = ipywidgets.HBox([sliders, selections, buttons, text_out])

display(ipywidgets.VBox([controls, image_out]))

def output_button_clicked(b):
    global i
    output_selected_spot(int(channels_text.value), int(spot1_text.value), i)
    if spot2_checked.value:
        spot2ID = int(spot2_text.value)
        output_selected_spot(int(channels_text.value), int(spot2_text.value), i)
    #output_selected_spot(channels_text.value, spot1_text.value, i)
    with text_out:
        text_out.clear_output()
        if spot2_checked.value:
            print("Saved spots %i and %i from channel %i in cell %i"%(int(spot1_text.value), int(spot2_text.value), int(channels_text.value), i))
        else:
            print("Saved spot %i from channel %i in cell %i and no others"%(int(spot1_text.value), int(channels_text.value), i))
        
def next_button_clicked(b):
    global i
    i = i + 1
    with text_out:
        text_out.clear_output()
        print("Iterated to cell %i, ready to display"%(i))
    find_spots(i)

output_button.on_click(output_button_clicked)
display_button.on_click(display_button_clicked)
next_cell_button.on_click(next_button_clicked)
